In [ ]:
import streamlit as st
import plotly.express as px
from plotly import tools
from plotly.graph_objs import *
import pandas as pd
import numpy as np
import os

from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Import weather station data and convert to dataframe

In [ ]:
#Read data from csv file
data = pd.read_csv('input/WeatherStation_Sample.dat', sep=',', header=1, skipinitialspace=True)
data.to_csv('input/WeatherStation_Sample.csv', encoding='utf-8', index=True)

#Set the index
data = data.set_index(data['TIMESTAMP'])
data.drop('TIMESTAMP', inplace=True, axis=1)
data.columns = ['Record', 'Battery Voltage', 'Machine Temperature', 'Air Temperature', 'Relative Humidity', 'Solar Irradiance (W)', 'Solar Irradiance (MJ)', 'Wind Speed', 'Wind Direction', 'Total Rain', 'Pressure']

#Create a new dataframe ignoring the Record, Battery voltage, and Machine temperature columns 
data_f = data[2:]
data_f.index = pd.to_datetime(data_f.index)

#Turn the values into floats as it was causing issues
for i in data[1:].keys():
    data_f = data_f.astype(float)

#Start date is set as 26/05/2022 11:40:00 which is when main data collection
start = pd.to_datetime("26-05-2022 11:40:00")
start_index = data_f.index.get_indexer([start], method='nearest')[0]

data_f = data_f[start_index:]
start_date, end_date = data.index[0], data.index[-1]

#desired_plot_count = len(data_f.keys()[1:]) #Uncomment this if you want that maximum amount of graphs

#3 graph plots were chosen for viewing experience but this can be editted as desired
desired_plot_count = 3

desired_plot_names = ['Air Temperature', 'Relative Humidity', 'Total Rain']

row_titles = []
for i in desired_plot_names:
    row_titles.append('(' + str(data[i].iloc[0]) + ')')

#Make the subplots (which all function as one big image realistically)
fig = make_subplots(rows=desired_plot_count, cols=1,
                    vertical_spacing=0.07,
                    subplot_titles = desired_plot_names,
                    shared_xaxes = True,
                    row_titles = row_titles
                   )
n = 1
for i in desired_plot_names:
    fig.add_trace(go.Scatter(x = data_f.index[-1008:], y=data_f[i][-1008:], name = i),
                  row=n, col=1)
    n += 1


fig.update_layout(height= 750, showlegend = False, margin=dict(l=5, r=10, t=60, b=5), font_size = 25)
fig.update_annotations(font_size=30)

#####

# Day to Day table creation

In [ ]:
#Make a function which converts compass degrees to descriptions
def deg2compass(num):
    val=int((num/22.5)+0.5)
    dir_list=["N","NNE","NE","ENE","E","ESE", "SE", "SSE","S","SSW","SW","WSW","W","WNW","NW","NNW"]
    return(dir_list[(val % 16)])

#Create data title column
test_df =  pd.DataFrame(data = np.array([["\U0001F321 Temperature"],
                                         ["\U0001F321 Previous 24 hour high/low"],
                                         ["\U0001F4A7 Relative Humidity"], ['\U0001F4A7 Rainfall'], ['\U0001F4A7 Chance of Rain'],
                                         ["\U000021F5 Pressure"],
                                         ["\U0001F4A8 Wind Speed"], ["\U0001F4A8 Wind Direction"],
                                         ["\U0001F325 Cloud Coverage"],
                                         ["\U0001F31E UV Index"],
                                         ["\U0001F33C Pollen"]]), columns = ['Most Recent Data:'])

#Create data values column
d2d_df_2 =  pd.DataFrame(data = np.array([[str('{:.1f}'.format(data_f['Air Temperature'].iloc[-1])) + "\U000000B0C"],
                                          ["Min: " + str('{:.1f}'.format(min(data_f['Air Temperature'].iloc[-24*6:-1]))) + "\U000000B0C, Max: " + str('{:.1f}'.format(max(data_f['Air Temperature'].iloc[-24*6:-1]))) + "\U000000B0C"],
                                          [str('{:.1f}'.format(data_f['Relative Humidity'].iloc[-1])) + "\U00000025"],
                                          [str('{:.1f}'.format(data_f['Total Rain'].iloc[-1])) + " mm"],
                                          ["— (N/A)"],
                                          [str('{:.0f}'.format(data_f['Pressure'].iloc[-1])) + " mbar"],
                                          [str('{:.0f}'.format(data_f['Wind Speed'].iloc[-1])) + " m/s"],
                                          [deg2compass(float((data_f['Wind Direction'].iloc[-1]))) + " (" + '{:.0f}'.format(data_f['Wind Direction'].iloc[-1]) + "\U000000B0)"],
                                          [str(26) + "\U00000025 (PH)"],
                                          ["— (N/A)"],
                                          ['Moderate' + " (PH)"]]), columns = [str(data_f.index[-1])])# Example dataframe

#Combine the two columns
d2d_df = pd.concat([test_df, d2d_df_2], axis = 1)
d2d_df

# Make AQI table

In [ ]:
#testing_df =  pd.DataFrame(data = np.array([[1.5], [4.5], [7.5], [9.5], [6.3]]).T) # Example dataframe for testing purposes

#Set the desired pollutant names
pollutants = ['PM10', 'PM2.5', 'NO2', 'O3']
p_subindex = []
for i in pollutants:
    p_subindex.append(str(i) + '_SubIndex')

#Make the base pollutants dataframe
pollutants_df = pd.DataFrame(
    columns=('ID', 'PM10', 'PM2.5', 'NO2', 'O3'))

#Set the ID column as the AQI and set it as the "Index" column
pollutants_df['ID'] = ['AQI', '\U000000B5g/m3']

#Read the values from the csv
aqi_df = pd.read_csv('input/aqi_df.csv', sep=',', header=0, skipinitialspace=True)
aqi_indexes_df = aqi_df[['PM10_SubIndex', 'PM2.5_SubIndex', 'NO2_SubIndex', 'O3_SubIndex']]

for i, j in zip(pollutants, p_subindex):
    pollutants_df[i] = pd.concat([aqi_df[j].iloc[-2:-1], aqi_df[i].iloc[-2:-1]], ignore_index=True)


# Working Dashboard below!!!

Mostly comprehensive examples can be found here: https://dash.plotly.com 

For example, I used this extensively: https://dash.plotly.com/datatable/conditional-formatting

In [ ]:
#Set up the Streamlit layout
#This replaces the Dash html.Div tree; st.columns reproduces the 60/40 split.

#Step 1) Pollutants Table (this is the worst part because it uses a lot of conditional formatting)
#Dash used style_data_conditional with filter_query; a pandas Styler is the equivalent here.
def aqi_colour(value):
    try:
        value = float(value)
    except (TypeError, ValueError):
        return ''
    #This next part says that if it is between the values of 0 and 4, set the background to green
    #The same idea is used for the other different bandings
    if 0 <= value < 4:
        return 'background-color: green; color: white'
    if 4 <= value < 7:
        return 'background-color: orange; color: white'
    if 7 <= value < 9:
        return 'background-color: red; color: white'
    if 9 <= value < 10:
        return 'background-color: black; color: white'
    return ''

def aqi_column_styles(column):
    styles = [''] * len(column)
    styles[0] = aqi_colour(column.iloc[0])
    return styles

#Position it so that it takes up the top left hand corner
col1, col2 = st.columns([6, 4])

with col1:
    st.dataframe(
        pollutants_df.style.apply(aqi_column_styles, subset=pollutants_df.columns[1:], axis=0),
        hide_index=True,
        width='stretch',
    )

#Step 2) Adds the Day to Day variables table on the right hand side
with col2:
    st.table(d2d_df)

#Step 3) Add the graph plots that were made at the start and positions them in the bottom left
st.plotly_chart(fig, width='stretch')

#Run with: streamlit run app.py


# Good luck :) It can be a pain to get it to work as desired, but when it works it is satisfying